## Step 5

Inputs: s3://thesis--ec331-s3/merged-price-volume-bids/
Output: s3://thesis--ec331-s3/capped-volume-bids/  

What does this do?  
This script corrects for the phenenomenon where some firms bid more than their max availability, and then rely on the dispatch algorithm to account for this, which means we can't rely

In [1]:
import pandas as pd
import numpy as np
import awswrangler as wr
import time
import re

print("Starting data load...")
start_time = time.time()

# Define S3 input and output paths
s3_input_path = "s3://thesis--ec331-s3/de-duped-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER1_202310010000.parquet/deduped.parquet"
s3_output_path = "s3://thesis--ec331-s3/step-2-capped-volume-bids/"

# Read Parquet file from S3
df = wr.s3.read_parquet(path=s3_input_path)
print(f"Data loaded in {time.time() - start_time:.2f} seconds")
print(f"DataFrame shape: {df.shape}")
print(f"Memory usage: {df.memory_usage().sum() / 1024 / 1024:.2f} MB")

# Show a sample of the raw data (adjust columns as needed)
print("\nSample data:")
columns_to_show = ["DUID", "TRADINGDATE", "PERIODID", "MAXAVAIL"] + \
                  [f"BANDAVAIL{i}" for i in range(1, 11)]
print(df[columns_to_show].sample(3))

def cap_row(row):
    """
    Caps the BANDAVAIL volumes for a single row so that the total volume 
    (when summed in order from BANDAVAIL10 down to BANDAVAIL1) does not exceed MAXAVAIL.
    
    Since there are no price bands in this dataset, the capping is applied directly to
    the volume bid columns, starting with BANDAVAIL10 and moving backwards.
    """
    max_avail = row["MAXAVAIL"]
    running_sum = 0.0
    row_capped = False  # Flag to indicate if any capping occurred
    
    # Process the volume bid columns in descending order: BANDAVAIL10 to BANDAVAIL1
    for i in range(10, 0, -1):
        col = f"BANDAVAIL{i}"
        current_volume = row.get(col, 0.0)
        remaining = max_avail - running_sum
        
        if remaining <= 0:
            # No capacity remaining: set this volume to zero
            if current_volume > 0:
                row_capped = True
            row[col] = 0.0
        else:
            # Cap the current volume if it exceeds the remaining capacity
            if current_volume > remaining:
                row_capped = True
            capped_volume = min(current_volume, remaining)
            row[col] = capped_volume
            running_sum += capped_volume
    
    # Optionally, record whether capping occurred for this row
    row["capped"] = row_capped
    return row

print("\nProcessing volume bids row by row (this may take a while)...")
# Process each row using the cap_row function
df = df.apply(cap_row, axis=1)

# Count the number of rows that were capped
total_capped_rows = df["capped"].sum()
# Optionally drop the temporary 'capped' column
df.drop(columns=["capped"], inplace=True)

print(f"\nTotal rows processed: {len(df)}")
print(f"Total rows capped: {total_capped_rows} ({total_capped_rows/len(df)*100:.2f}%)")
print(f"\nProcessing completed in {time.time() - start_time:.2f} seconds")
print("\nSample of capped DataFrame:")
print(df.head())

# Extract the date part from the input path to build the output folder name
date_match = re.search(r'(\d{12})', s3_input_path)
date_str = date_match.group(1) if date_match else "processed"
output_folder = f"{s3_output_path}RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_{date_str}_capped/"

print(f"\nWriting results to {output_folder}")
wr.s3.to_parquet(
    df=df,
    path=output_folder,
    dataset=True,
    mode='overwrite',
    compression='snappy'
)

print(f"Process completed in {time.time() - start_time:.2f} seconds")
print(f"Results written to {output_folder}")
print(f"Summary: {total_capped_rows} out of {len(df)} rows were capped ({total_capped_rows/len(df)*100:.2f}%)")

Starting data load...
Data loaded in 0.43 seconds
DataFrame shape: (41760, 28)
Memory usage: 8.92 MB

Sample data:
           DUID TRADINGDATE  PERIODID  MAXAVAIL  BANDAVAIL1  BANDAVAIL2  \
6291   ASRMGE02  2023-10-09     244.0       1.0         1.0         0.0   
20642    VENUS1  2023-10-11     195.0      20.0         0.0         0.0   
5061   WALGRVG1  2023-10-08     166.0       0.0         0.0         0.0   

       BANDAVAIL3  BANDAVAIL4  BANDAVAIL5  BANDAVAIL6  BANDAVAIL7  BANDAVAIL8  \
6291          0.0         0.0         0.0         0.0         0.0         0.0   
20642         0.0         5.0         5.0        10.0         0.0         0.0   
5061          0.0         0.0         0.0         0.0         0.0         0.0   

       BANDAVAIL9  BANDAVAIL10  
6291          0.0          0.0  
20642         0.0          8.0  
5061          0.0         27.0  

Processing volume bids row by row (this may take a while)...

Total rows processed: 41760
Total rows capped: 27434 (65.69%)

P